In [0]:
path = "/Volumes/workspace/vol_project/raw/csv/"

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, DoubleType

schema = StructType([
    StructField("open_time", LongType()),
    StructField("open", DoubleType()),
    StructField("high", DoubleType()),
    StructField("low", DoubleType()),
    StructField("close", DoubleType()),
    StructField("volume", DoubleType()),
    StructField("close_time", LongType()),
    StructField("quote_asset_volume", DoubleType()),
    StructField("number_of_trades", LongType()),
    StructField("taker_buy_base_volume", DoubleType()),
    StructField("taker_buy_quote_volume", DoubleType()),
    StructField("ignore", DoubleType()),
])

In [0]:
df = spark.read.csv(path, header=False, schema=schema)

In [0]:
display(df)

In [0]:
df.count()

In [0]:
from pyspark.sql.functions import col, current_timestamp

bronze_df = (
    df
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("ingested_at", current_timestamp())
)

display(bronze_df)

In [0]:
   bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.vol_project.bronze_btc_1m")

In [0]:
spark.table("workspace.vol_project.bronze_btc_1m").count()

In [0]:
spark.table("workspace.vol_project.bronze_btc_1m").count()

In [0]:
from pyspark.sql.functions import count

display(
    spark.table("workspace.vol_project.bronze_btc_1m")
    .groupBy("source_file")
    .agg(count("*").alias("rows"))
    .orderBy("source_file")
)

In [0]:
from pyspark.sql.functions import col

df = spark.table("workspace.vol_project.bronze_btc_1m")
print("millisecond rows:", df.filter(col("open_time") < 1e15).count())
print("microsecond rows:", df.filter(col("open_time") >= 1e15).count())